
# GSOD Weather Data — SQL Overview

**Dataset:** `bigquery-public-data.noaa_gsod`  
**Primary tables:**
- `gsod*` (one table per year, e.g., `gsod2019`, `gsod2020`, …) — daily observations
- `stations` — station metadata (name, country, state, lat, lon, elevation, active period)

This notebook gives you structured prompts and blank cells to practice SQL in BigQuery.



## Why this dataset?
It’s simple but realistic, supports **joins**, and includes **data-quality issues** to clean:

- Date is split across `year`, `mo`, `da` (often typed as STRING in some yearly tables).  
- Missing values are represented by **sentinel numbers**, e.g. `9999.9`, `999.9`, `99.99`, (varies by column).  
- Join keys: `stn` (USAF) and `wban` identify a station and can be joined to `stations` (use both keys when available).


## Getting started
Run this notebook in any of the options from class 3 (see `03-Data-Warehousing/README.md`). The solutions use the `%%bigquery` magic. In a BigQuery notebook with SQL cells, leave that line out.

Try each question on your own first. If you get stuck, uncomment the `disp(...)` line in a HINT or SOLUTION cell and run it.

In [ ]:
# Run this cell to enable the hints and solutions
from IPython.display import Pretty as disp
hint = 'https://raw.githubusercontent.com/soltaniehha/Business-Analytics-Toolbox/master/docs/hints/'  # path to hints on GitHub

**Running this in free Colab?** Free Colab doesn't know which Google Cloud project to bill, so run the cell below before any query. Uncomment the lines and put your project ID in place of `your-project-id`. Skip it in BigQuery Studio, Workbench, and Colab Enterprise.

In [ ]:
# Free Colab only: sign in and set the project your queries bill to
# from google.colab import auth
# auth.authenticate_user()
# %env GOOGLE_CLOUD_PROJECT=your-project-id


---
## A) Basic exploration (work on a single year, e.g., `gsod2019`)
**Goal:** get familiar with the raw table.
### 1. Preview 5–10 rows of `bigquery-public-data.noaa_gsod.gsod2019`.

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-preview')

### 2. List all columns and confirm data types.

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-columns')

### 3. Count total rows and number of distinct stations (`stn`, `wban`) for 2019.


- **`stn`** → Station number (a global identifier).  
- **`wban`** → Weather Bureau Air Force Navy number (mainly U.S. identifier).

Neither is unique on its own. Some stations share the same `stn` with different `wban` values, and others have no `wban`.
To ensure a unique station key across all years and countries, we combine them:

```sql
CONCAT(stn, '-', wban) AS station_id
```


In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-row-count')

### 4. Find the top 10 stations by number of reported days in 2019.

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-top-stations')


---
## B) Construct a proper DATE

### 1. build a `date` column as a true `DATE` from `year`, `mo`, `da`.
Using `DATE(CAST(year AS INT64), CAST(mo AS INT64), CAST(da AS INT64)) AS date`, select a few rows to verify.


In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-date')

### 2. Create a **temporary cleaned subquery** (via `WITH`) that
* replaces `year`, `mo`, and `da` with `date`
* replaces `stn` and `wban` with our new `station_id`

In [ ]:
# HINT: Uncomment and execute the cell below to get help
#disp(hint + '06-01-cte-hint')

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-cte')

## C) Handle missing values (sentinel codes)

### C.1 For columns like `dewp`, `slp`, `stp`, `visib`, `wdsp`, `mxpsd`, `gust`, `max`, `min`, `prcp`, find the sentinel value reported in the column description.

As noted in the notebook's introduction, common sentinel codes are `9999.9`, `99.99`, `999.9` (which varies by column). The best way to do this is to check the [original documentation](https://www.ncei.noaa.gov/data/global-summary-of-the-day/doc/readme.txt) and then verify.

*   `dewp`, `slp`, `max`, `min`: `9999.9`
*   `sndp`, `stp`, `wdsp`, `mxpsd`, `gust`, `visib`: `999.9`
*   `prcp`: `99.99`

The documentation gives `9999.9` for `stp`, but this table uses `999.9`.

### C.2 Count how many rows hit each sentinel value in 2019 (one query per column or a union).
Tip: Use the function `COUNTIF()`

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-sentinel-count')

### C.3 In a `WITH` subquery, `NULLIF` those sentinels. Let's also include the `station_id` and `date` columns we created earlier. Make sure to include all the other columns that we didn't treat so we don't lose any of the variables.

Additionally, `wdsp` and `mxpsd` were initially stored as `STRING` types, which is problematic for numerical analysis. Let's fix this by casting them to `FLOAT64` and then applying `NULLIF`:

```sql
NULLIF(CAST(wdsp AS FLOAT64), 999.9) AS wdsp,
NULLIF(CAST(mxpsd AS FLOAT64), 999.9) AS mxpsd,
```

Note: `NULLIF` helps you replace a specific value with **NULL**.

In [ ]:
# HINT: Uncomment and execute the cell below to get help
#disp(hint + '06-01-nullif-hint')

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-nullif')

### C.4 Verify by re-counting NULLs after cleaning.

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-null-count')

---

## D) Join with `stations` metadata

### D.1 Preview a few rows of `bigquery-public-data.noaa_gsod.stations`.

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-stations-preview')

### D.2 Join your cleaned 2019 observations to `stations` on `station_id` where possible, and select relevant columns and in this order:

```date, station_id, name, country, state, lat, lon, elev, temp, dewp, max, min, prcp, wdsp```



In [ ]:
# HINT: Uncomment and execute the cell below to get help
#disp(hint + '06-01-join-hint')

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-join')

### D.3 Count how many 2019 rows have **no match** in `stations` (left join) — report the count.

In [ ]:
# HINT: Uncomment and execute the cell below to get help
#disp(hint + '06-01-no-match-hint')

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-no-match')

---

## E) Create a partitioned **clean** table

### E.1 Create a partitioned table `temp_dataset.gsod_2019_clean` with cleaned data.

Use your D.2 join, partitioned by `date`. If your project doesn't have a `temp_dataset` dataset yet, create it first, in the US location.

In [ ]:
# HINT: Uncomment and execute the cell below to get help
#disp(hint + '06-01-clean-table-hint')

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-clean-table')

### E.2 Confirm row count after table creation.

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-table-count')

Observation: We lose 10697 points after the join. These are weather reports for stations that we don't have name or other pieces of information in the stations table.

---

## F) Intro summaries

### F.1 Average `temp` by month for 2019 (ignore NULLs).

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-avg-temp')

### F.2 Number of rainy days per station (`prcp > 0`) in 2019; show the top 10 stations.

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-rainy-days')

### F.3 By country: average `max` and `min` temperature for 2019; keep only countries with at least 200 station-days (`HAVING`).

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-country-temp')

---

## G) Quality checks

### G.1 Show min/max of `temp`, `max`, `min`, `prcp` after cleaning.

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-min-max')

### G.2 Count rows where `max < min` (should be rare or zero).

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-max-below-min')

### G.3 Flag any station with `lat` or `lon` outside realistic bounds (after join).

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-lat-lon')

---

## H) Bonus question

### H.1 Create a simple **UDF** (BigQuery SQL) that classifies a day as `'freezing'`, `'cool'`, `'warm'`, `'hot'` based on `temp`.

Use these cutoffs in °F: 32 and below is freezing, up to 60 is cool, up to 80 is warm, above 80 is hot.

In [ ]:
# HINT: Uncomment and execute the cell below to get help
#disp(hint + '06-01-udf-hint')

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-udf')

### H.2 Use your function on a few rows of `temp_dataset.gsod_2019_clean`.

In [ ]:
# SOLUTION: Uncomment and execute the cell below to get help
#disp(hint + '06-01-udf-use')